# Analise de populacao 2010 e 2022

Este notebook le a planilha do projeto, agrega a populacao por estado e organiza estados e municipios pelo maior crescimento entre 2022 e 2010.


In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
xlsx_path = base_dir / 'CD2022_Populacao_2010_Compatibilizada_20231222.xlsx'
estado_csv = base_dir / 'populacao_agregada_por_estado_crescimento_2022_2010.csv'
municipio_csv = base_dir / 'populacao_por_municipio_crescimento_2022_2010.csv'


In [ ]:
df = pd.read_excel(xlsx_path, sheet_name=0, header=2)
df = df.dropna(how='all').copy()
df.head()


In [ ]:
col_uf = 'UF'
col_cod_uf = 'COD. UF'
col_cod_municipio = 'COD. MUNIC'
col_municipio = 'NOME DO MUNICIPIO'
col_pop_2010_sinopse = 'Populacao Municipio 2010 (Sinopse)'
col_pop_2010_compat = 'Populacao 2010 (Alteracoes de Limites ate 2022)1'
col_pop_2022 = 'Populacao Censo 2022'

df.columns = [
    '',
    col_uf,
    col_cod_uf,
    col_cod_municipio,
    col_municipio,
    col_pop_2010_sinopse,
    col_pop_2010_compat,
    col_pop_2022,
]
df = df.drop(columns=[''])

for col in [col_pop_2010_sinopse, col_pop_2010_compat, col_pop_2022]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=[col_uf, col_municipio, col_pop_2010_compat, col_pop_2022])

nomes_saida = {
    col_uf: 'UF',
    col_cod_uf: 'COD_UF',
    col_cod_municipio: 'COD_MUNIC',
    col_municipio: 'NOME_DO_MUNICIPIO',
    col_pop_2010_sinopse: 'populacao_municipio_2010_sinopse',
    col_pop_2010_compat: 'populacao_2010_compatibilizada',
    col_pop_2022: 'populacao_censo_2022',
}


In [ ]:
populacao_por_estado = (
    df.groupby(col_uf)[[col_pop_2010_sinopse, col_pop_2010_compat, col_pop_2022]]
    .sum()
    .reset_index()
)
populacao_por_estado['crescimento_2022_2010'] = populacao_por_estado[col_pop_2022] - populacao_por_estado[col_pop_2010_compat]
populacao_por_estado = populacao_por_estado.sort_values('crescimento_2022_2010', ascending=False).reset_index(drop=True)
populacao_por_estado = populacao_por_estado.rename(columns=nomes_saida)
populacao_por_estado.to_csv(estado_csv, sep=';', index=False)
populacao_por_estado.head(10)


In [ ]:
populacao_por_municipio = df[[
    col_uf,
    col_cod_uf,
    col_cod_municipio,
    col_municipio,
    col_pop_2010_sinopse,
    col_pop_2010_compat,
    col_pop_2022,
]].copy()
populacao_por_municipio['crescimento_2022_2010'] = populacao_por_municipio[col_pop_2022] - populacao_por_municipio[col_pop_2010_compat]
populacao_por_municipio = populacao_por_municipio.sort_values('crescimento_2022_2010', ascending=False).reset_index(drop=True)
populacao_por_municipio = populacao_por_municipio.rename(columns=nomes_saida)
populacao_por_municipio.to_csv(municipio_csv, sep=';', index=False)
populacao_por_municipio.head(10)
